# Metric Ablations

In the following we demonstrate how to reproduce the metric ablations that show the complementary value of normality and independence testing in SITN.

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

from sitn.aggregators import MaxQuantile
from sitn.datasets.cifar10c import CORRUPTIONS
from sitn.utils import construct_results_path

In [ ]:
# Configurations
# We assume the model has already be trained and evaluated with
# these configurations (follow the cross-dataset and perturbation
# OOD detection notebooks to see how).

train_cfg = {"dataset_name": "cifar10"}

eval_cfg_train = {"config": train_cfg, "split_pick": "train"}
eval_cfg_val = {"config": train_cfg, "split_pick": "val"}
eval_cfg_test = {"config": train_cfg, "split_pick": "test"}

eval_cfgs_ood = [
    {
        "config": train_cfg,
        "eval_dataset_name": "cifar10c",
        "corruptions": [corruption],
        "split_pick": "test",
    }
    for corruption in CORRUPTIONS
]

In [ ]:
# Load train and val ID predictions
id_train_preds = pd.read_csv(construct_results_path(**eval_cfg_train, result_type="predictions"))
id_val_preds = pd.read_csv(construct_results_path(**eval_cfg_val, result_type="predictions"))

# Fit SITN
sitn = MaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
sitn.fit(id_val_preds)

In [ ]:
# Metric configurations
metrics = {
    "sitn": {"label": "SITN", "higher_is_ood": True},
    "anderson_darling_statistic": {"label": "Anderson Darling", "higher_is_ood": True},
    "ps_cv": {"label": "Power Spectrum CV", "higher_is_ood": True},
}

# Load ID test predictions
id_preds = pd.read_csv(construct_results_path(**eval_cfg_test, result_type="predictions"))
id_preds["train_dataset"] = eval_cfg_test["config"]["dataset_name"]
id_preds["eval_dataset"] = eval_cfg_test["config"]["dataset_name"]

results = []
for eval_cfg_ood in eval_cfgs_ood:
    # Load OOD test predictions
    ood_preds = pd.read_csv(construct_results_path(**eval_cfg_ood, result_type="predictions"))
    ood_preds["train_dataset"] = eval_cfg_ood["config"]["dataset_name"]
    ood_preds["eval_dataset"] = eval_cfg_ood["eval_dataset_name"]

    # Combine ID and OOD predictions
    preds = pd.concat([id_preds.copy(), ood_preds], ignore_index=True)

    # Add SITN scores
    preds["sitn"] = sitn.score(preds)

    # Compute AUROC for each method
    y_true = (preds["eval_dataset"] != preds["train_dataset"]).astype(int)
    for col, meta in metrics.items():
        scores = preds[col].copy()
        if not meta["higher_is_ood"]:
            scores = -scores

        auroc = roc_auc_score(y_true, scores)
        results.append(
            {
                "corruption": eval_cfg_ood["corruptions"][0],
                "metric": meta["label"],
                "AUROC": auroc,
            }
        )

results = pd.DataFrame(results).pivot(index="corruption", columns="metric", values="AUROC")
results = results.reindex(columns=[meta["label"] for meta in metrics.values()])
results


metric,SITN,Anderson Darling,Power Spectrum CV
corruption,,,
brightness,0.820263,0.850395,0.524225
contrast,0.599640,0.601770,0.574987
defocus_blur,0.731072,0.523879,0.755502
elastic_transform,0.737486,0.614149,0.763748
fog,0.635524,0.489668,0.674187
frost,0.884353,0.794549,0.837835
gaussian_blur,0.751795,0.583029,0.786091
gaussian_noise,0.907656,0.692877,0.922047
glass_blur,0.953968,0.850579,0.954321
